In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, ReLU
import joblib
import os

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/Metro_Interstate_Traffic_Volume.csv")
df['date_time'] = pd.to_datetime(df['date_time'])
df.set_index('date_time', inplace=True)

# Encode categorical features
le_holiday = LabelEncoder()
df['holiday'] = le_holiday.fit_transform(df['holiday'])

le_weather = LabelEncoder()
df['weather_main'] = le_weather.fit_transform(df['weather_main'])
df['weather_description'] = le_weather.fit_transform(df['weather_description'])

# Fill missing values
df.fillna(0, inplace=True)

# Features and target
features = ['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description']
target = ['traffic_volume']

X = df[features].values
y = df[target].values

# Optional: log transform target to stabilize variance
y_log = np.log1p(y)

# Normalize
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y_log)

# models directory
if not os.path.exists('models'):
    os.makedirs('models')

# Save scalers and encoders
joblib.dump(scaler_X, "models/scaler_X.pkl")
joblib.dump(scaler_y, "models/scaler_y.pkl")
joblib.dump(le_holiday, "models/le_holiday.pkl")
joblib.dump(le_weather, "models/le_weather.pkl")

# Create sequences
SEQ_LENGTH = 10
def create_sequences(X, y, seq_length):
    X_seq, y_seq = [], []
    for i in range(len(X)-seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X_scaled, y_scaled, SEQ_LENGTH)

# Split train/test
train_size = int(len(X_seq)*0.8)
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
y_train, y_test = y_seq[:train_size], y_seq[train_size:]

# Build LSTM
model = Sequential()
model.add(LSTM(64, activation='relu', input_shape=(SEQ_LENGTH, X_seq.shape[2])))
model.add(Dense(1))
model.add(ReLU())  # ensures output >= 0

model.compile(optimizer='adam', loss='mse')

# Train
model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.1)

# Save model in native Keras format (.keras)
model.save("models/traffic_lstm_model.keras")
print("Model saved as models/traffic_lstm_model.keras")

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1085/1085 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - loss: 0.7719 - val_loss: 0.7870
Epoch 2/30
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.7719 - val_loss: 0.7870
Epoch 3/30
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 0.7713 - val_loss: 0.7870
Epoch 4/30
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.7708 - val_loss: 0.7870
Epoch 5/30
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.7697 - val_loss: 0.7870
Epoch 6/30
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.7708 - val_loss: 0.7870
Epoch 7/30
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.7699 - val_loss: 0.7870
Epoch 8/30
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.7708 - val_loss: 0.7870
Epoch 9/30
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.7688 - val_loss: 0.7870
Epoch 10/30
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.7710 - val_loss: 0.7870
Epoch 11/30
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.7709 - val_loss: 0.7870
Epoch 12/30
1085/1085 ━━━━━━━━━━━━━━━━━